# MaizeCare: maize leaf disease and severity

This notebook brings the whole project together **without training anything**. It reads the trained models and
their results from Google Drive and runs the same code as the training scripts and the web app (the `maize`
package from GitHub).

1. **Data**: photos per class, near-copy groups, train / val / test split
2. **Stage 1, classifier**: which disease? (four backbones compared, MobileNetV2 chosen)
3. **Leaf mask**: GrabCut grown from the middle of the photo
4. **Stage 2, autoencoders**: how severe? (trained on healthy leaves only; the error is the severity)
5. **Test results** against the proposal's targets
6. **The full pipeline** on test photos, and on your own photo

A CPU runtime is enough.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q "maize[data,serve] @ git+https://github.com/Kos-en/maizecare.git@mask-and-severity"

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from maize.pipeline import load_manifest, select
from maize.preprocess import CLS_SIZE, decode, leaf_fraction, prepare
from maize.serve import Scorer, onnx_runner

ROOT = Path("/content/drive/MyDrive")       # manifest.csv, the photos and the image caches
MODELS = ROOT / "maize_models"
CLASSIFIER = MODELS / "clf_mobilenet_v2"    # the chosen classifier
RUN = MODELS / "v2"                         # the autoencoder run with the GrabCut leaf mask
AE = next(f for f in sorted(RUN.glob("ae_*")) if (f / "metrics.json").exists())  # the one maize-evaluate chose
print("classifier:", CLASSIFIER.name, "| autoencoder:", AE.name)

## 1. Data

Every photo goes through the data pipeline first (`maize-prepare`). Near copies (close on both pHash and aHash)
are grouped, groups holding two classes are dropped, and whole groups are split 70 / 15 / 15, so the same leaf can
never be in both training and test.

In [ ]:
df = load_manifest(ROOT)
usable = select(df)
print(f"{len(df)} photos, {len(usable)} usable")
display(df.groupby(["label", "status"]).size().unstack(fill_value=0))
display(usable.groupby(["label", "split"]).size().unstack(fill_value=0))

## 2. Stage 1: the classifier

Four ImageNet backbones, transfer learning, the same split. Target (a): within 3 points of ResNet50 and at least
3x smaller. MobileNetV2 is chosen.

In [ ]:
def classifier_row(folder: Path) -> dict:
    info = json.loads((folder / "classifier.json").read_text())
    return {"backbone": info["backbone"], "val_macro_f1": info["val"]["macro_f1"],
            "test_macro_f1": info["test"]["macro_f1"], "test_ece": info["test"]["ece"],
            "size_mb": (folder / "classifier.onnx").stat().st_size / 1e6}


classifiers = pd.DataFrame([classifier_row(f) for f in sorted(MODELS.glob("clf_*"))])
display(classifiers.sort_values("test_macro_f1", ascending=False).round(3))

In [ ]:
info = json.loads((CLASSIFIER / "classifier.json").read_text())
labels, matrix = info["labels"], np.array(info["test"]["confusion_matrix"])

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap="Greens")
ax.set(xticks=range(len(labels)), xticklabels=labels, yticks=range(len(labels)), yticklabels=labels,
       xlabel="predicted", ylabel="true", title=f"{info['backbone']}: test photos")
for (true, predicted), n in np.ndenumerate(matrix):
    ax.text(predicted, true, n, ha="center", va="center")
plt.show()
display(pd.DataFrame(info["test"]["per_class"]).T.loc[labels, ["precision", "recall", "f1-score", "support"]].round(3))

## 3. The leaf mask

The autoencoder must only look at the photographed leaf. GrabCut learns the leaf's colours from the middle of the
photo (the farmer aims at the leaf) and the background's colours from the rest; the piece holding the middle is kept.
An earlier colour rule (Excess Green) kept weeds and dropped yellowed leaves; replacing it raised the val AUROC
from 0.65 to 0.86. Masks covering under 10% or over 95% of the photo are flagged as doubtful and left out.

In [ ]:
def read_rgb(path: str) -> np.ndarray:
    return cv2.cvtColor(cv2.imread(str(ROOT / path)), cv2.COLOR_BGR2RGB)


def outline(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """The photo dimmed outside the leaf, with the mask edge in magenta."""
    shown = (image * 0.35).astype(np.uint8)
    shown[mask == 1] = image[mask == 1]
    edges, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return cv2.drawContours(shown, edges, -1, (255, 0, 255), 2)


test = select(df, split="test")
examples = test.groupby("label").sample(2, random_state=1)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, leaf) in zip(axes.T.ravel(), examples.iterrows()):   # one column per class
    image, mask = prepare(read_rgb(leaf["path"]), CLS_SIZE)
    ax.imshow(outline(image, mask))
    ax.set_title(f"{leaf['label']}: leaf {leaf_fraction(mask):.0%} of photo")
    ax.axis("off")
plt.show()

## 4. Stage 2: the autoencoders

A convolutional autoencoder and a VAE are trained on **healthy leaves only**, inside the leaf mask. A diseased leaf
rebuilds badly, so its error is the severity signal: no severity labels needed. Each is scored on val with the plain
error and a smoothed one (colour, not fine texture). The pair with the best AUROC on its **weakest** disease is
chosen, because target (b) is per disease. (Computed here from the saved files, so this notebook needs no Keras.)

In [ ]:
rows = [{"autoencoder": f.parent.name, "error": error, **result}
        for f in sorted(RUN.glob("ae_*/validation.json"))
        for error, result in json.loads(f.read_text()).items()]
autoencoders = pd.DataFrame(rows)
autoencoders["worst_auroc"] = autoencoders.filter(like="auroc_").min(axis=1)
display(autoencoders.sort_values("worst_auroc", ascending=False).drop(columns="healthy_median_error").round(3))

## 5. Test results against the targets

The test photos were used once, after every choice was made on val.

* **(b)** AUROC (healthy vs each disease) >= 0.80
* **(c)** Spearman with the symptomatic area A(x) >= 0.40 for MLND and MLB, and better than both baselines.
  A(x) = % of the leaf in a symptom colour range (yellowing, brown lesions, dead tissue), like Leaf Doctor / ASSESS.

In [ ]:
metrics = json.loads((AE / "metrics.json").read_text())
results = pd.DataFrame({
    disease: {"AUROC": r["auroc_vs_healthy"], "Spearman vs A(x)": r["spearman_vs_area"],
              **{f"baseline: {name}": value for name, value in metrics["baselines"][disease].items()}}
    for disease, r in metrics["diseases"].items()}).T
results["(b) AUROC >= 0.80"] = results["AUROC"] >= 0.80
results["(c) Spearman >= 0.40"] = results["Spearman vs A(x)"] >= 0.40
display(results.round(3))
print(f"error measured: {metrics['error']}")
print(f"healthy test leaves scored as unusually high: {metrics['false_high_rate']:.1%}")
print("doubtful masks:", {label: f"{share:.0%}" for label, share in metrics["doubtful_mask_share"].items()})

In [ ]:
display(Image(str(AE / "error_by_class.png")))
display(Image(str(AE / "heatmaps.png")))   # input inside the mask, reconstruction, where the error is

## 6. The full pipeline

Exactly what the web app does for an uploaded photo: the classifier names the disease from the whole photo, the
leaf mask is found, the autoencoder rebuilds the leaf, and its error inside the mask becomes a 0-1 severity score
(0 = like a typical healthy leaf, 1 = among the worst diseased leaves seen).

In [ ]:
classes = json.loads((CLASSIFIER / "classifier.json").read_text())
calibration = json.loads((AE / "calibration.json").read_text())
scorer = Scorer(onnx_runner(CLASSIFIER / "classifier.onnx"), onnx_runner(AE / "model.onnx"), classes, calibration)


def show(rgb: np.ndarray, title: str = "") -> dict:
    result = scorer.score(rgb)
    image, mask = prepare(rgb, CLS_SIZE)
    plt.figure(figsize=(4, 4))
    plt.imshow(outline(image, mask))
    plt.title(f"{title}\n{result['label']} ({result['confidence']:.0%}), severity {result['score']:.2f}")
    plt.axis("off")
    plt.show()
    return result


for _, leaf in test.groupby("label").sample(1, random_state=7).iterrows():
    show(read_rgb(leaf["path"]), f"true: {leaf['label']}")

Try your own photo (centre the leaf in the picture):

In [ ]:
from google.colab import files

for name, data in files.upload().items():
    show(decode(data), name)

## Limitations and next steps

* Detection (b) is met; severity ranking (c) is partial: the autoencoder follows streaks and lesions (MSV, MLB) better
  than broad yellowing (MLND), which it partly rebuilds.
* Some photos in the Healthy folder are not maize (e.g. bean leaves): to be cleaned.
* Next: a colour-reconstruction autoencoder (grey leaf in, colour out) for MLND, human by-eye severity ratings,
  confidence intervals, and results per data source.